## Findings

- `AgentResponseUpdate` are sent as `WorkflowEvents`.
- A downstream executor receives the completed `AgentExecutorResponse`, not streaming.
- The agent's update events are identified by its `executor_id`
- Agents run concurrently within the same superstep, so their `AgentResponseUpdate` messages can be interlaced
- Workflows do not need converge after a `fan-out`. Each executor terminating executor just receives its final values.
- Multiple `send_message()` downstream invokes that downstream executor once for each message. It will all process within the same `super_step`. In other words, the `super_step` does not end until all executors have completed (which includes all of their `send_message()` calls)
- MAF is based on a Pregel / Bulk Synchronous Parallel (BSP) Graph. It is a **not** a data transformation pipeline. Each node wakes-up and does something independently (if needed). Therefore, you need to think of the workflow as a big node-centrice graph processor, where communication moves through the graph, but what is computed and output is based by each node (executor) independently.

In [ ]:
from agent_framework import Agent, WorkflowBuilder, executor, WorkflowContext, AgentExecutorResponse
from agent_framework.openai import OpenAIChatClient
import os
from dotenv import load_dotenv

load_dotenv()

chat_client = OpenAIChatClient(
    base_url=os.getenv("FOUNDRY_OPENAI_BASE_URL"),
    api_key=os.getenv("FOUNDRY_API_KEY"),
    model=os.getenv("FOUNDRY_MODEL"),
)

poem_agent = Agent(
    client=chat_client,
    name="Poem Agent",
    description="An agent that generates poems.",
    instructions="Generate poems based on the given prompts.",
    tools=[],
    context_providers=[],
)

lyric_agent = Agent(
    client=chat_client,
    name="Lyric Agent",
    description="An agent that generates song lyrics.",
    instructions="Generate song lyrics based on the given prompts.",
    tools=[],
    context_providers=[],
)


In [ ]:
@executor(id="upper_case_executor")
async def upper_case(value: str, ctx: WorkflowContext[str, str]) -> None:
# async def upper_case(value: AgentExecutorResponse, ctx: WorkflowContext[str, str]) -> None:
    """Convert the input to uppercase and forward it to the next node."""
    # await ctx.send_message(value.agent_response.text.upper())
    await ctx.send_message(value.upper())
    await ctx.yield_output("Converting text to uppercase") # WorkflowEvent(type='output', executor_id='upper_case_executor', data='Converting text to uppercase')


@executor(id="reverse_executor")
async def reverse(value: str, ctx: WorkflowContext[str, str]) -> None:
# async def reverse(value: AgentExecutorResponse, ctx: WorkflowContext[str, str]) -> None:
    """Reverse the input text and forward it to the next node."""
    # await ctx.send_message(value.agent_response.text[::-1])
    await ctx.send_message(value[::-1])
    await ctx.yield_output("Reversing text") # WorkflowEvent(type='output', executor_id='reverse_executor', data='Reversing text')
    await ctx.yield_output("Reversing text 2") # WorkflowEvent(type='output', executor_id='reverse_executor', data='Reversing text')
    await ctx.yield_output("Reversing text 3") # WorkflowEvent(type='output', executor_id='reverse_executor', data='Reversing text')


@executor(id="start_node")
async def start_node(prompt: str, ctx: WorkflowContext[str, str]) -> None:
    """Convert the input to lowercase and forward it to the next node."""
    await ctx.send_message("Something short about a quiet life")
    await ctx.send_message("Something short about a hotdog")
    await ctx.yield_output("Starting execution") # WorkflowEvent(type='output', executor_id='start_node', data='Starting execution')

In [34]:
workflow = (
    WorkflowBuilder(start_executor=start_node)
    .add_fan_out_edges(start_node, [upper_case, reverse])
    # .add_edge(poem_agent, upper_case)
    # .add_edge(lyric_agent, upper_case)
).build()
 
async for event in workflow.run("Write about a quiet life. Make it short.", stream=True):
    # LEFT-OFF: what happens when I send multiple messages downstream?
    # LEFT-OFF: changing this to show how multiple outputs are received and stored with the executor
    print(event)
    # if event.type == 'output' and isinstance(event.data, AgentResponseUpdate):
    #     print(f"({datetime.now()}) Agent Update: {event.data.text}")

WorkflowEvent(type='started')
WorkflowEvent(type='status', state=IN_PROGRESS)
WorkflowEvent(type='superstep_started', iteration=1)
WorkflowEvent(type='executor_invoked', executor_id='start_node', data='Write about a quiet life. Make it short.')
WorkflowEvent(type='output', executor_id='start_node', data='Starting execution')
WorkflowEvent(type='executor_completed', executor_id='start_node', data=['Something short about a quiet life', 'Something short about a hotdog', 'Starting execution'])
WorkflowEvent(type='superstep_completed', iteration=1)
WorkflowEvent(type='superstep_started', iteration=2)
WorkflowEvent(type='executor_invoked', executor_id='upper_case_executor', data='Something short about a quiet life')
WorkflowEvent(type='output', executor_id='upper_case_executor', data='Converting text to uppercase')
WorkflowEvent(type='executor_completed', executor_id='upper_case_executor', data=['SOMETHING SHORT ABOUT A QUIET LIFE', 'Converting text to uppercase'])
WorkflowEvent(type='execut